# <u>Group S</u>: Anne Faury, Héloise Lordez, Cyprien Charlaté, Lucas Gorrec, William Roose


# Week 2 — Baby Names: Initial Implementation

This notebook explore three interactive visualisations made with **Altair 6** exploring the French baby-names dataset (department agregation, year-by-year, 1900-2020). We implement here our three favorite designs chosen during the first week. The given notebook from the 'Names hints' folder and previous class labs on Altair give us some inspiration and tools to build the current notebook.

To get a further understanding and explanations about our results, check our post on the forum on Moodle.

### Reminder:
<u>Visualization 1</u>: 
- How do baby names evolve over time?
- Are there names that have consistently remained popular or unpopular?
- Are there some that have were suddenly or briefly popular or unpopular?
- Are there trends in time?

<u>Visualization 2</u>: 
- Is there a regional effect in the data? 
- Are some names more popular in some regions? 
- Are popular names generally popular across the whole country?

<u>Visualization 3</u>: 
- Are there gender effects in the data? 
- Does popularity of names given to both sexes evolve consistently? (Note: this data set treats sex as binary; this is a simplification that carries into this assignment but does not generally hold.)


> **Data notes**  
> - Rows with `preusuel == "_PRENOMS_RARES"` (rare names) are dropped.  
> - Rows with `dpt == "XX"` (unknown department) and `annais == "XXXX"` (unknown year) are dropped.  
> - `sexe`: 1 = male, 2 = female.

In [ ]:
import altair as alt
import pandas as pd
import geopandas as gpd

# html renderer outputs interactive Vega-Embed HTML (text/html) that VS Code renders natively.
# disable_max_rows inlines all data so no external file references are needed.
alt.data_transformers.disable_max_rows()
alt.renderers.enable('html')

DATA  = "../Names hints/dpt2020.csv"
GEO_M = "../Names hints/departements-avec-outre-mer.geojson"

# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA, sep=";", dtype={"dpt": str, "annais": str})
df = df[
    (df.preusuel != "_PRENOMS_RARES") &
    (df.dpt      != "XX")             &
    (df.annais   != "XXXX")
]
df["annais"] = df["annais"].astype(int)
df["nombre"] = df["nombre"].astype(int)
df["sexe"]   = df["sexe"].astype(int)

print(f"Rows: {len(df):,}  |  Unique names: {df.preusuel.nunique():,}  "
      f"|  Years: {df.annais.min()}–{df.annais.max()}  "
      f"|  Departments: {df.dpt.nunique()}")
df.head(3)

## Visualization 1 — How do baby names evolve over time? (V2)

**Design (updated after peer feedback):** A multi-line time-series chart, now built around two complementary interactions instead of a single static top-20:

- **Sliding top-20 (decade slider):** instead of one top-20 fixed over 1900–2020 (which structurally favours old, long-lived names), the top-20 is now recomputed over a **10-year sliding window**, moved one year at a time via a slider. Moving the slider changes *which* 20 names are drawn, while the x-axis always stays the full 1900–2020 range — so once a name is "in", you see its entire trajectory, not just the slice where it peaked. This directly answers "are there names that were suddenly or briefly popular" by letting that brief peak surface its own top-20 window.
- **Free-text search bar:** a search box overlays the trajectory of **any single name in the dataset** — not just the top-20 — typed in any case (e.g. "kamal", "Karen", "ALICE"). This was the most consistent piece of feedback (Kamal, Karen, Alice) and removes the "names outside the top-20 are invisible" limitation entirely.
  - **Exact match, not partial:** the overlay only appears once the typed text matches a name *in full* (case-insensitive). An early partial-match version showed every name containing the typed substring while still typing (e.g. "ale" briefly matched Alexandre, Valentine, etc. before the word was finished), which was confusing and not what was being searched for.
  - **Black dashed line, not a colour:** the searched name is drawn in solid black with a dashed stroke, deliberately outside the categorical palette used for the top-20. An earlier version used a fixed red, which turned out to be visually close to one of the reds already used by the top-20 colour scale — the dash pattern adds a second, palette-independent way to tell the two apart.
  - **In-place label instead of a legend entry:** because the search line's colour is fixed rather than mapped to a colour encoding, Vega-Lite cannot list it in the colour legend. Instead, the matched name is printed as a bold text label directly at the end of its own curve (its most recent data point), which also scales better than a legend if multiple searches were ever shown at once.
- **Softer highlight:** clicking a name in the legend still highlights it, but the dimmed opacity for the other lines was raised from 0.07 to 0.30 (Julien's feedback) so the surrounding context (relative scale, co-occurring trends) doesn't disappear completely.

**What we kept on purpose:** legend-first click-to-highlight (intuitive, no extra UI), and the full 1900–2020 time axis (the period itself is the whole point of the question).

**Strengths:** removes the fixed-top-20 temporal bias; any name is reachable; less aggressive dimming preserves context.
**Trade-offs:** the sliding top-20 is recomputed on a *national* aggregate only (not per-region or per-sex) to keep the interaction simple; with very large numbers of distinct names appearing across all decades, the legend can still get visually dense — search remains the more precise tool for a specific name.

**Performance fix (search bar):** with ~15,000 distinct first names over 121 years, a naive "long" table (one row per name x year, ~250k rows) made the search re-evaluate a string match on every row, on every keystroke — this caused visible lag and eventually crashed the rendering engine after pressing Enter. The data was restructured into one row per name (years/counts stored as lists), with the match test run once per name and the per-year detail unfolded only afterwards (`transform_filter` -> `transform_flatten`) — cutting both the number of filter evaluations (~16x) and the JSON payload size (~4x, since the name string is no longer repeated on every row). The later move to exact-match search (see above) made this even more effective: most keystrokes now match zero names, so the filter does very little work until the full name is typed.

**Known trade-off — shared Y-axis between the top-20 and the search overlay:** the chart uses a single Y-scale shared across all three layers (top-20, search line, search label), and that scale's domain is recomputed from whatever is currently visible — which includes the active top-20 window. This means that moving the decade slider while a search is active can shift the Y-axis range slightly, even though the searched name's underlying data never changes. In practice the effect is minor (the top-20's own range is fairly stable across windows — the same handful of very large "classic" names tend to set the upper bound regardless of the window), so the searched curve doesn't visibly distort, but its position can drift a little.
We considered two fixes — giving each layer its own independent Y-scale (`resolve_scale(y="independent")`), or fixing the Y-domain to a precomputed global maximum — but both trade a small, rare cosmetic issue for a worse one: independent scales would require two Y-axes on the same plot (confusing to read), and a fixed global maximum could be set by an obscure name found only through search, needlessly compressing the much more commonly viewed top-20 curves. We chose to keep the shared, auto-ranging scale and accept the minor axis drift as a known limitation, rather than degrade the default reading experience to fix an edge case.

In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
national   = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()

YEAR_MIN, YEAR_MAX = int(national.annais.min()), int(national.annais.max())
WINDOW = 10  # taille de la fenêtre glissante (en années)

# Top-20 glissant : pour chaque décennie possible (pas de 1 an), on calcule le
# top-20 sur la fenêtre [start, start+WINDOW-1]. On stocke, pour chaque nom,
# la LISTE des décennies où il apparaît dans ce top-20 (colonne "decades_arr",
# une vraie liste -> Altair la sérialise en array JS, pas en string).
pivot_v1 = national.pivot_table(index="preusuel", columns="annais", values="nombre", fill_value=0)
for y in range(YEAR_MIN, YEAR_MAX + 1):
    if y not in pivot_v1.columns:
        pivot_v1[y] = 0
pivot_v1 = pivot_v1[sorted(pivot_v1.columns)]

decade_starts_v1 = list(range(YEAR_MIN, YEAR_MAX - WINDOW + 2))
name_to_decades_v1 = {}
for start in decade_starts_v1:
    window_sum = pivot_v1[list(range(start, start + WINDOW))].sum(axis=1)
    for name in window_sum.nlargest(20).index:
        name_to_decades_v1.setdefault(name, []).append(start)

# On garde aussi le top-20 "toute période" (comme la V1 originale) pour ne
# jamais perdre les grands classiques même s'ils sortent ponctuellement
# du top-20 d'une fenêtre donnée.
top20_overall_v1 = national.groupby("preusuel")["nombre"].sum().nlargest(20).index.tolist()
for name in top20_overall_v1:
    name_to_decades_v1.setdefault(name, [])

names_needed_v1 = sorted(name_to_decades_v1.keys())
decades_compact_v1 = pd.DataFrame({
    "preusuel":    list(name_to_decades_v1.keys()),
    "decades_arr": [sorted(v) for v in name_to_decades_v1.values()],
})

# data_v1 : 1 ligne = 1 (nom, année) pour les noms qui apparaissent dans AU
# MOINS une fenêtre top-20 -> trace complète 1900-2020 (cf. justification).
data_v1 = (
    national[national.preusuel.isin(names_needed_v1)]
    .merge(decades_compact_v1, on="preusuel", how="left")
)

# data_search_v1 : TOUT le dataset national, pour la barre de recherche libre
# (n'importe quel prénom, même hors top-20, doit pouvoir être superposé).
#
# Format COMPACT : 1 ligne par prénom (années + comptes stockés en listes),
# au lieu d'1 ligne par (prénom, année). Avec ~15 000 prénoms distincts sur
# 121 ans, le format "long" produit ~250k lignes ; le test de correspondance
# de la recherche serait alors réévalué jusqu'à 121 fois pour le même prénom
# à chaque frappe clavier, ce qui sature le moteur de rendu (observé : lag
# puis crash après validation de la saisie). Le format compact réduit le
# nombre d'évaluations du filtre d'un facteur ~16x et la taille JSON
# transportée d'un facteur ~4x (le nom n'est plus répété sur chaque ligne).
# Le détail année par année est restitué via transform_flatten, appliqué
# SEULEMENT APRES le filtre (donc seulement sur les noms qui matchent).
data_search_v1 = (
    national.sort_values(["preusuel", "annais"])
    .groupby("preusuel")
    .agg(years=("annais", list), counts=("nombre", list))
    .reset_index()
)

print(f"data_v1: {len(data_v1):,} lignes | {data_v1.preusuel.nunique()} prénoms (apparus dans >=1 top-20 glissant)")
print(f"data_search_v1: {len(data_search_v1):,} lignes (1/prénom) | {data_search_v1.preusuel.nunique()} prénoms (recherche libre)")

# ── Params interactifs ─────────────────────────────────────────────────────────
decade_p1 = alt.param(
    name="v1_decade",
    bind=alt.binding_range(
        min=YEAR_MIN, max=YEAR_MAX - WINDOW + 1, step=1,
        name=f"Fenêtre top-20 ({WINDOW} ans) — année de début : ",
    ),
    value=YEAR_MIN,
)
search_p1 = alt.param(
    name="v1_search",
    bind=alt.binding(input="search", placeholder="Tape un prénom en entier (ex : Kamal)…",
                      name="Rechercher un prénom : "),
    value="",
)
sel = alt.selection_point(fields=["preusuel"], bind="legend")

in_window_v1 = "indexof(datum.decades_arr, v1_decade) >= 0"

# Match EXACT (insensible à la casse), pas de match partiel : on n'affiche
# une courbe de recherche que lorsque la saisie correspond entièrement à un
# prénom du dataset. Ce choix règle, par construction, le problème de
# plusieurs courbes qui apparaissaient pendant la frappe (ex: "ale" matchait
# Alexandre, Valentine... avant même la fin de la saisie).
search_active1 = "length(v1_search) > 0"
search_match1  = "upper(datum.preusuel) == upper(v1_search)"
search_filter1 = f"{search_active1} && {search_match1}"

# Domaine X étendu de quelques années au-delà de YEAR_MAX, pour laisser de la
# place à l'étiquette texte du prénom recherché (collée à la fin de la
# courbe) sans qu'elle ne dépasse du cadre — fréquent puisque beaucoup de
# prénoms recherchés sont encore "actifs" en 2020, donc leur dernier point
# tombe exactement sur le bord droit de l'axe.
X_DOMAIN_V1 = [YEAR_MIN, YEAR_MAX + 6]
x_enc_v1 = alt.X("annais:Q", title="Year",
                  scale=alt.Scale(domain=X_DOMAIN_V1),
                  axis=alt.Axis(format="d", tickCount=13))

# ── Couche 1 : top-20 glissant (recalculé selon la fenêtre sélectionnée) ──────
top20_layer_v1 = (
    alt.Chart(data_v1)
    .mark_line()
    .transform_filter(in_window_v1)
    .encode(
        x=x_enc_v1,
        y=alt.Y("nombre:Q", title="Number of births (national)"),
        color=alt.Color("preusuel:N",
                         legend=alt.Legend(title="Click a name to highlight",
                                            symbolStrokeWidth=3)),
        # Opacité du dim remontée de 0.07 -> 0.30 (retour Julien) : on garde
        # plus de contexte sur les 19 autres lignes pendant un highlight.
        opacity=alt.condition(sel, alt.value(1.0), alt.value(0.30)),
        strokeWidth=alt.condition(sel, alt.value(3.0), alt.value(1.2)),
        tooltip=[
            alt.Tooltip("preusuel:N", title="Name"),
            alt.Tooltip("annais:Q",   title="Year",   format="d"),
            alt.Tooltip("nombre:Q",   title="Births", format=","),
        ],
    )
    .add_params(sel, decade_p1)
)

# ── Couche 2 : courbe de la recherche (noir, pointillé) ──────────────────────
# Style délibérément hors palette catégorielle (noir + pointillé = double
# signal forme+couleur) pour ne jamais se confondre avec les 20 lignes du
# top-20, quelle que soit la palette de couleurs utilisée par ailleurs.
search_line_v1 = (
    alt.Chart(data_search_v1)
    .mark_line(color="black", strokeDash=[6, 3], strokeWidth=3)
    .transform_filter(search_filter1)
    .transform_flatten(["years", "counts"])
    .encode(
        x=alt.X("years:Q", scale=alt.Scale(domain=X_DOMAIN_V1)),
        y=alt.Y("counts:Q"),
        tooltip=[
            alt.Tooltip("preusuel:N", title="Name (search)"),
            alt.Tooltip("years:Q",    title="Year", format="d"),
            alt.Tooltip("counts:Q",   title="Births", format=","),
        ],
    )
    .add_params(search_p1)
)

# ── Couche 3 : étiquette texte avec le nom recherché, collée au dernier point ─
# La recherche n'a pas d'entrée dans la légende couleur (sa couleur est fixe,
# pas mappée à un encodage) -> on identifie le nom directement sur le graphe,
# via un label texte positionné sur le point le plus récent de sa courbe.
search_label_v1 = (
    alt.Chart(data_search_v1)
    .transform_filter(search_filter1)
    .transform_flatten(["years", "counts"])
    .transform_window(
        rank="rank()",
        sort=[alt.SortField("years", order="descending")],
        groupby=["preusuel"],
    )
    .transform_filter("datum.rank == 1")  # ne garde que le dernier point (année max)
    .mark_text(align="left", dx=8, dy=-4, fontWeight="bold", fontSize=12, color="black")
    .encode(
        x=alt.X("years:Q", scale=alt.Scale(domain=X_DOMAIN_V1)),
        y=alt.Y("counts:Q"),
        text="preusuel:N",
    )
)

v1 = (
    alt.layer(top20_layer_v1, search_line_v1, search_label_v1)
    .resolve_scale(color="independent")
    .properties(
        title=alt.TitleParams(
            "National births per year (1900-2020) — Sliding top-20 + free search",
            subtitle="Slider recomputes the top-20 over a 10-year window. Search overlays any single name from the dataset (exact match), in or out of the top-20.",
            subtitleColor="gray",
            subtitleFontSize=11,
        ),
        width=800,
        height=430,
    )
    .configure_view(strokeWidth=0)
)

v1


## Visualization 2 — Is there a regional effect? (V2)

**Design (updated after peer feedback):** Same overall layout as before — three small top-3 maps plus one large map — extended with two independent improvements:

- **Colour-scale toggle (auto / fixed) — scoped to the search overlay only:** a radio toggle switches the search overlay's colour domain between **auto** (rescaled to the searched name's values for the current year) and **fixed** (pinned to that name's own all-time maximum across every department and year). This directly matches the brief's stated purpose for "fixed" — "to observe a name rise or fade geographically as the year slider moves" — which is about tracking *one* name over time.

The toggle deliberately does **not** apply to the top-3 panels (the three medal mini-maps and the large map's top-3 view), which always auto-scale. We initially applied the same toggle everywhere, pinning the top-3 to the single dataset-wide maximum (~324‰, set by "Marie" in 1902). Testing on the real INSEE data showed this made every year from the 1970s onward look almost uniformly pale — that one historical peak is 7-10x higher than any top-3 value from the past 50 years, so the entire colour range available to recent decades was compressed into a sliver near the bottom of the scheme, with departments barely distinguishable from each other. This wasn't a rendering bug: it was a real but unhelpful consequence of applying "fixed" somewhere the brief never asked for it. The top-3 panels show three *different* names every year by construction (whoever ranks 1st/2nd/3rd that year) — there is no single name to "track over time" there, so pinning them to an all-time ceiling answers a question ("how does this year's leader compare to the single most concentrated name in history") the brief doesn't pose for that view. Restricting the toggle to the search overlay — where a stable per-name scale genuinely serves the stated purpose — keeps the top-3 panels readable in every decade while still giving the search feature the comparability it needs.

**Bug found along the way:** an intermediate version shared one colour-scale expression across all three colour definitions (mini-maps, large top-3 map, search overlay), with a branch reading the searched name to set the ceiling. That accidentally coupled the top-3 mini-maps to whatever was typed in the search box — typing "Jacques" (own max ~43‰) silently capped the *unrelated* top-3 mini-maps at 43 too, clipping their real values (which could exceed 100‰) hard past the top of the colour scheme and rendering several departments near-black. Confirmed by checking the actual underlying data (Finistère/Jacques/1944 is genuinely 17.2‰, well inside a 0-40 legend — the ~110 value reported was from the unrelated, wrongly-capped top-3 mini-map) and fixed by making the top-3 colour definitions structurally incapable of referencing the search parameter at all, rather than relying on conditional logic to keep them apart.
- **Free-text search (top 1 000 names) replacing the top-3-only constraint:** the original top-3 stayed as the **national reference point** — it directly answers "are popular names popular everywhere?", which a pure search box alone could not. A free-text search box was added *alongside* it: typing a name fills the large map with that name's regional distribution instead, while the three small maps keep showing the current year's top-3 untouched. Search takes priority over the top-3 click while it has text; clearing the search box returns to whichever top-3 panel was last clicked. We initially considered a true "whichever was interacted with last wins" rule, but this requires a hand-written Vega event-stream signal outside Altair's normal declarative parameter API — judged too fragile to maintain for the benefit, so we kept the simpler, fully predictable rule instead.

**Why top 1 000, not the full ~15 000 names:** a truly unrestricted search would need every (department x name x year) combination available client-side. Even sparse (only non-zero values), we measured this at roughly 2 million rows for the full dataset — about 8x larger than the ~250k-row table that already caused lag and a render crash in Visualization 1 before that fix. Restricting the search to the 1 000 most common names nationally (all-time) keeps the payload in the same order of magnitude as the optimised Viz-1 search while covering far more ground than the original top-3.

**Implementation note — avoiding geometry duplication in the search layer:** naively joining department geometry to a (department x name) search table would duplicate every department's polygon once per searchable name (up to 1000x) — exactly the blow-up the existing top-3 wide-pivot (`p_YYYY` columns) was already designed to avoid. Instead, the search table keeps **exactly one row per department** (never duplicated), with a nested object `{name: {years: [...], props: [...]}}` holding every top-1000 name with at least one non-zero value there. Vega's expression language supports dynamic key access on such an object (`datum.names_data[v2_search]`), so the matching name's series is looked up directly in-browser without ever turning department x name into rows.

**What we kept on purpose:** the 1‑per-1000-births normalisation, the small-multiples-plus-large-map layout, and the Corsica/overseas-territory handling — all unchanged and not contested in feedback.

**Strengths:** colour comparability is now an explicit choice rather than a hidden default; any of the 1 000 most common names can be explored regionally, well beyond the original top-3; the national top-3 reference is preserved for the "popular everywhere?" question.
**Trade-offs:** search is capped at the top 1 000 names (a rare or very localised name outside that list cannot be searched) — a deliberate volume/coverage compromise, not an oversight; returning from a search to the top-3 view requires clearing the search box rather than a single click on a mini-map.

**Tested on the real INSEE dataset — accent sensitivity.** Running this on the actual `dpt2020.csv` (not the synthetic data used during development) surfaced a case the synthetic data never exercised: about 13% of the top-1 000 names contain an accented letter (e.g. STÉPHANE, ANDRÉ, FRANÇOIS), and Vega's expression language has no built-in accent-insensitive comparison (`String.prototype.normalize()` is not available — only a fixed catalogue of functions is). Typing "stephane" therefore does not match "STÉPHANE". We considered stripping accents on both sides (search input and stored keys) to fix this, but found 27 pairs of names in the top-1 000 that differ *only* by an accent and are recorded as genuinely distinct names by INSEE (e.g. LÉA vs LEA, JÉRÉMY vs JEREMY, KÉVIN vs KEVIN) — merging them would silently combine two different historical counts into one. We chose to keep them distinct and leave the search accent-sensitive, and adjusted the "no match" message accordingly (`"No exact match — check spelling/accents"` rather than `"Not in top 1 000"`, which would have been misleading for a correctly-listed name typed without its accent).

**Bug fix — case-sensitive search:** the first working version silently failed for any name not typed in uppercase exactly (e.g. typing "Jean" showed nothing, even though "JEAN" is in the dataset), because looking up a name inside the nested `names_data` object is exact-match by construction — there is no implicit case folding on object-key access the way there is on string comparison. Fixed by normalising the typed name with `upper()` before the lookup (`datum.names_data[upper(v2_search)]`), the same fix already applied in Visualization 1's search.

**Bug fix — blank departments that were actually zero, not missing:** comparing the same name/year side-by-side on the top-3 panels and the search overlay (e.g. "Jean" in 1975) revealed departments rendered pure white on the search map that were pale blue on the top-3 map — a visible inconsistency for identical underlying data. The cause: the search data only stores years where a name had at least one birth (a sparse representation, kept deliberately compact — see above). A department with zero "Jean" births in 1975 specifically (but some in other years) has no 1975 entry in its `years` list, so the lookup returned "not found" (`idx == -1`); the original code then filtered that row out entirely, making the department disappear from the map rather than showing a true zero. The top-3 panels never had this problem because their wide pivot table (`p_YYYY` columns) is explicitly zero-filled for every year. Fixed by mapping a missing year to 0 instead of dropping the row (`datum.idx >= 0 ? ... : 0`), so a department with no recorded births that year is shown as pale blue (a real zero, inside the colour scale) — visually consistent with the top-3 panels — while a name entirely absent from the top-1000 list still correctly renders an empty map (no rows at all, not a zero).

**How this design evolved — what we tried first, and why it changed.** Our first version pinned "fixed" mode to a single all-time, all-names maximum, shared by the top-3 panels and the search overlay alike, reasoning that a shared ceiling would let the colour intensity itself reveal that name concentration has declined across the decades. On synthetic test data this looked reasonable. Testing on the real INSEE data exposed two real problems with it, in order:
1. A genuine bug: a shared colour-scale expression let the search box's chosen name silently override the *unrelated* top-3 panels' ceiling too (see the bug note above).
2. Once that was fixed and the top-3 panels were correctly pinned to their own intended shared ceiling, a second, more fundamental issue surfaced: that ceiling (~324‰, set by "Marie" in 1902) is so much higher than any top-3 value from the past 50 years that every recent year rendered almost uniformly pale, with departments barely distinguishable from each other — "fixed" mode became close to unusable for exactly the years most people would want to explore.

Rather than patch this further (e.g. a percentile-based ceiling, or a log scale), we went back to the brief's own description of what "fixed" is for — tracking one name's footprint over time — and recognised the top-3 panels don't fit that description at all, since they show three *different* names every year. The toggle now applies only where it was actually meant to: the search overlay.

**Two follow-up refinements:**
- **Explicit "not in top 1 000" message.** Searching a name outside the searchable list previously rendered a silent, unexplained blank map — indistinguishable at a glance from "this name has zero births everywhere". The header now checks whether the typed name exists in the per-name maximum lookup (the same list backing the search data) and, if not, shows "Not in top 1 000 — unavailable" in red instead of the name.
- **Shared colour scale across the three medal panels.** The three small maps (rank 1/2/3) previously each rescaled independently in "auto" mode, which could make the third-place name look just as visually "dark" as the first-place one even though its absolute values are lower — undermining the very point of ranking them. They now follow the same auto/fixed toggle as the large map: in "auto", all three rescale together to their combined range (so a real popularity gap between rank 1 and rank 3 is now visible as a real colour gap); in "fixed" this changes nothing, since all three already shared the same all-time maximum.

**Real bug found and fixed — search box was silently capping the mini-maps' colour scale.** Testing on the real INSEE data surfaced a genuine bug, not just a misreading: typing a name in the search box (e.g. "Jacques") would also visibly darken/clip the three *top-3 mini-maps* (Jean/Marie/Michel), even though they have nothing to do with the search. The cause: an earlier version used a single shared expression for all three colour definitions (mini-maps, large top-3 map, search overlay), with one branch reading `v2_search` to pick the ceiling. That branch was meant for the search overlay only, but the mini-maps' colour definition (`COLOR_SMALL`) referenced the very same expression, so it silently inherited the searched name's own (often much smaller) maximum as its ceiling too — e.g. a top-3 mini-map value of ~110 rendered against a ceiling of ~43 (Jacques' own max) clips hard past the top of the "blues" scheme, which is what produced the near-black departments. The fix splits this into two independent expressions: one used only by the mini-maps (`_DOMAIN_MAX_EXPR_TOP3`, which never references `v2_search` at all), and one shared by the large map's two mutually-exclusive layers (`_DOMAIN_MAX_EXPR_LARGE`, used by both the top-3 view and the search view of the large map, which is what lets them share a single legend rather than stacking two — Vega-Lite renders a declared legend even when its layer currently has no visible data, so two separate scale objects here would always show two overlapping legends). After the fix, typing any name in the search box no longer affects the mini-maps' colours in any way.

**Missing legend on the mini-maps, fixed.** The three medal mini-maps never had a colour legend, even in the very first version of this visualization — fine when each one was just a small preview, but no longer once they were given a deliberately *shared* colour scale (so the real popularity gap between rank 1/2/3 shows up as a real colour gap, see above). Without a legend there was no way to read what a given shade of blue actually meant before clicking through to the large map — whose own legend, in search mode, shows a *different* scale entirely (the searched name's own maximum, not the top-3's shared one). A single legend now appears next to the three mini-maps: declaring it on all three looks like it should triple it, but because they share one resolved colour scale (`resolve_scale(color="shared")`), Vega-Lite merges the three declarations into one legend next to the row, exactly as it would for any other shared-scale view.

**Clarifying subtitle for the search overlay's scale.** Comparing a "fixed" search result (e.g. Charles, own peak ~54‰) against the top-3 mini-maps just above it (shared peak ~324‰) side by side, with no further context, risks reading Charles as uniformly faint everywhere — when "auto" mode shows real regional variation, just on a scale roughly 6x smaller, since Charles never came close to the top-3's historical peak. A small italic subtitle now appears under the searched name, stating the exact ceiling in force: "Scale: 0–54‰ — this name's own all-time peak" in fixed mode, or "Scale: auto-adjusted to this year's range" in auto mode. The number shown is read from the same lookup driving the colour scale itself, so it can't drift out of sync with what's actually rendered.

In [ ]:
import warnings
import math

# ── 1. National top-3 per year (unchanged — kept as the national reference) ──
nat_v2 = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()
nat_v2["rank_id"] = (
    nat_v2.groupby("annais")["nombre"]
    .rank(method="first", ascending=False)
    .astype(int)
)
top3_yr       = nat_v2[nat_v2["rank_id"] <= 3][["preusuel", "annais", "rank_id"]].copy()
top3_names_v2 = top3_yr["preusuel"].unique().tolist()

# ── 2. Dept proportions (per 1 000 births) — top-3 only, as before ───────────
by_dpt = (
    df[df["preusuel"].isin(top3_names_v2)]
    .groupby(["preusuel", "dpt", "annais"], as_index=False)["nombre"].sum()
)
tot_dpt = (
    df.groupby(["dpt", "annais"], as_index=False)["nombre"]
    .sum().rename(columns={"nombre": "total"})
)
by_dpt = by_dpt.merge(tot_dpt, on=["dpt", "annais"])
by_dpt["prop"] = (by_dpt["nombre"] / by_dpt["total"] * 1_000).round(1)
data_long_v2 = (
    top3_yr.merge(by_dpt, on=["preusuel", "annais"], how="left")
    .fillna({"prop": 0.0})
)

# ── 3. Wide pivot: 1 row per (dept x rank_id), columns p_YYYY ────────────────
def rank_wide(rid):
    sub = data_long_v2[data_long_v2["rank_id"] == rid]
    w = (
        sub.pivot_table(index="dpt", columns="annais", values="prop", aggfunc="first")
        .fillna(0)
    )
    w.columns = [f"p_{int(y)}" for y in w.columns]
    w = w.reset_index()
    w["rank_id"] = rid
    return w

wide_all_v2 = pd.concat([rank_wide(1), rank_wide(2), rank_wide(3)], ignore_index=True)

# INSEE CSV codes Corsica as "20"; GeoJSON uses "2A"/"2B"
_corse = wide_all_v2[wide_all_v2["dpt"] == "20"].copy()
if len(_corse) > 0:
    _c2a = _corse.copy(); _c2a["dpt"] = "2A"
    _c2b = _corse.copy(); _c2b["dpt"] = "2B"
    wide_all_v2 = pd.concat(
        [wide_all_v2[wide_all_v2["dpt"] != "20"], _c2a, _c2b],
        ignore_index=True,
    )

# ── 4. Merge geometry — DOM-TOM repositioned + latitude correction ────────────
from shapely.affinity import translate as _sha_translate, scale as _sha_scale

depts_geo_v2 = gpd.read_file(GEO_M)

_inset_cfg = {
    "971": (-4.0, 40.6, 0.8),
    "972": (-1.5, 40.6, 0.6),
    "973": ( 1.5, 40.4, 0.8),
    "974": ( 6.0, 40.6, 0.8),
    "976": ( 8.8, 40.6, 0.5),
}

def _reposition(gdf, cfg):
    rows = []
    for _, row in gdf.iterrows():
        if row["code"] not in cfg:
            rows.append(row)
            continue
        b = row.geometry.bounds
        orig_size = max(b[2] - b[0], b[3] - b[1])
        tx, ty, tsize = cfg[row["code"]]
        factor = tsize / orig_size
        g = _sha_scale(row.geometry, xfact=factor, yfact=factor, origin="centroid")
        b2 = g.bounds
        g = _sha_translate(g, tx - (b2[0]+b2[2])/2, ty - (b2[1]+b2[3])/2)
        row = row.copy()
        row["geometry"] = g
        rows.append(row)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)

depts_geo_v2 = _reposition(depts_geo_v2, _inset_cfg)

_LAT_CORR = 1.0 / math.cos(math.radians(46.0))
depts_geo_v2["geometry"] = depts_geo_v2.geometry.apply(
    lambda g: _sha_scale(g, xfact=1.0, yfact=_LAT_CORR, origin=(0, 0))
)
depts_geo_v2 = depts_geo_v2.set_geometry("geometry")

geo_v2 = depts_geo_v2.merge(wide_all_v2, how="inner", left_on="code", right_on="dpt")
geo_v2 = geo_v2.drop(columns=["dpt"])

global_max_v2 = float(data_long_v2["prop"].max())
name_src_v2   = top3_yr[["preusuel", "annais", "rank_id"]].copy()
print(f"geo_v2: {len(geo_v2)} rows | top-3 colour domain [0, {global_max_v2:.1f}]")

# ── 5. Free-text search data (top-1000 national names, beyond the top-3) ────
# A truly unrestricted search (~15,000 names x ~96 depts x 121 years) would be
# far too large to ship to the browser, even in a compact format (we measured
# ~177M cells dense / ~2M rows sparse for the full dataset — well beyond what
# crashed the renderer in Visualization 1 at ~250k rows). As a deliberate
# trade-off, the search is restricted to the top 1 000 most popular names
# nationally (all-time total births), which keeps the payload to a comparable
# order of magnitude as the Viz-1 fix while covering far more than the top-3.
#
# Critically, the data is shaped so the MAP GEOMETRY IS NEVER DUPLICATED:
# instead of one row per (department x name) merged with geometry (which
# would multiply the department polygons up to 1000x — exactly what the
# wide p_YYYY pivot above already avoids for the top-3), each row stays
# ONE PER DEPARTMENT, with a nested object {name: {years: [...], props: [...]}}
# holding every top-1000 name that has at least one non-zero value there.
# Vega's expression language supports dynamic key access on such an object
# (datum.names_data[v2_search]), so the matching name's series is looked up
# directly in-browser without ever joining department x name as rows.
TOP_N_SEARCH_V2 = 1000
nat_totals_v2  = df.groupby("preusuel")["nombre"].sum().sort_values(ascending=False)
top1000_names_v2 = nat_totals_v2.head(TOP_N_SEARCH_V2).index.tolist()

by_dpt_search_v2 = (
    df[df["preusuel"].isin(top1000_names_v2)]
    .groupby(["preusuel", "dpt", "annais"], as_index=False)["nombre"].sum()
    .merge(tot_dpt, on=["dpt", "annais"])
)
by_dpt_search_v2["prop"] = (by_dpt_search_v2["nombre"] / by_dpt_search_v2["total"] * 1_000).round(1)

def _build_names_data(group):
    out = {}
    for name, sub in group.groupby("preusuel"):
        out[name] = {"years": sub["annais"].tolist(), "props": sub["prop"].tolist()}
    return out

search_compact_v2 = (
    by_dpt_search_v2.groupby("dpt")
    .apply(_build_names_data, include_groups=False)
    .reset_index(name="names_data")
)

# Same Corsica fix as the top-3 table, applied to the search table.
_corse_s = search_compact_v2[search_compact_v2["dpt"] == "20"].copy()
if len(_corse_s) > 0:
    _cs_2a = _corse_s.copy(); _cs_2a["dpt"] = "2A"
    _cs_2b = _corse_s.copy(); _cs_2b["dpt"] = "2B"
    search_compact_v2 = pd.concat(
        [search_compact_v2[search_compact_v2["dpt"] != "20"], _cs_2a, _cs_2b],
        ignore_index=True,
    )

geo_search_v2 = depts_geo_v2.merge(search_compact_v2, how="inner", left_on="code", right_on="dpt")
geo_search_v2 = geo_search_v2.drop(columns=["dpt"])
print(f"geo_search_v2: {len(geo_search_v2)} rows (1 per department, never duplicated by name) "
      f"| {len(top1000_names_v2)} searchable names")

# Per-name maximum proportion (across all years and departments for that name),
# used ONLY for the search overlay's "fixed" colour scale — see rationale below.
name_max_lookup_v2 = (
    by_dpt_search_v2.groupby("preusuel")["prop"].max().round(1).to_dict()
)

# ── 6. Params ─────────────────────────────────────────────────────────────────
year_p_v2 = alt.param(
    name="v2_year",
    bind=alt.binding_range(min=1900, max=2020, step=1, name="Year : "),
    value=2000,
)
click_sel_v2 = alt.selection_point(
    fields=["rank_id"],
    name="v2_click",
    empty=False,
    value=[{"rank_id": 1}],
)
search_p_v2 = alt.param(
    name="v2_search",
    bind=alt.binding(input="search", placeholder="Type a full first name (top 1 000 most common)…",
                      name="Search any name : "),
    value="",
)
# Toggle for the colour scale: "auto" rescales to the data currently on
# screen (maximises contrast for a single year/name, the previous default,
# kept as default here too); "fixed" pins the scale to the global max
# (computed once over the whole period) so that colours stay comparable
# across years/names — useful to watch a name's footprint genuinely grow or
# shrink as the year slider moves, rather than being re-normalised every
# frame. Both reviewers and our own check (rendering the same data under
# each mode) confirmed "auto" can make two years look equally dark even
# when the underlying values differ by an order of magnitude.
scale_mode_p_v2 = alt.param(
    name="v2_scale_mode",
    bind=alt.binding_radio(options=["auto", "fixed"], name="Colour scale : "),
    value="auto",
)
# Carries the per-name maximum lookup (computed in step 5 above) as a plain
# JS object, so it can be indexed by key inside an expression
# (v2_name_max_lookup[upper(v2_search)]) — same dynamic-key-access pattern
# used for names_data, but at the parameter level rather than per data row,
# since a colour scale's domain cannot reference a per-row datum directly.
name_max_lookup_p_v2 = alt.param(
    name="v2_name_max_lookup",
    value=name_max_lookup_v2,
)

# ── 7. Layout constants ───────────────────────────────────────────────────────
SW, SH = 205, 220
LW, LH = 530, 565

# A SINGLE colour scale definition, shared by the mini maps, the large top-3
# map, and the search overlay. In "fixed" mode its ceiling depends on context:
#   - no active search -> the shared top-3 maximum (global_max_v2, computed
#     once over the whole period). Keeping this shared across the three
#     medal panels is what lets the colour intensity itself show that name
#     concentration has declined over the decades (a real historical pattern,
#     not a scaling artefact) — recalculating it per name here would erase
#     that signal entirely, which is the opposite of what "fixed" is for.
#   - an active search -> that specific name's OWN all-time maximum
#     (name_max_lookup_v2[name]), so the chosen name's geographic footprint
#     can be read on its own scale as it grows or shrinks across the year
#     slider, rather than being dwarfed by comparison to old, highly
#     concentrated classics it was never going to approach.
#
# DESIGN CHANGE — the auto/fixed toggle now applies ONLY to the search
# overlay, not to the top-3 (mini-maps + large top-3 map). Earlier testing
# on the real INSEE data showed the global all-time maximum (1902, ~324‰) is
# 7-10x higher than any top-3 value from the 1970s onward — pinning the
# top-3 to that ceiling left every recent year looking almost uniformly pale
# ("washed out"), with departments barely distinguishable from one another.
# This isn't a rendering bug: it's a real consequence of the design brief's
# own stated purpose for "fixed" — "to observe a name rise or fade
# geographically as the year slider moves" — which describes tracking ONE
# name over time, not comparing three different yearly champions against a
# single all-time ceiling. The top-3 panels change WHO they show every year
# by construction, so a shared all-time ceiling answers a question ("how
# does this year's leader compare to the most concentrated name in history")
# that the brief never actually asked for the top-3 view. The top-3 now
# always auto-scales (maximising contrast for whichever three names are
# shown that year), while the search overlay keeps the toggle, which is
# where "track one name's own footprint over time on a stable scale"
# genuinely applies.
#
# (Earlier bug fix, still relevant for the search overlay's branch: an
# expression shared across all three colour definitions once let the search
# box's chosen name silently cap the unrelated top-3 mini-maps' ceiling too.
# Keeping the top-3 expression independent of v2_search avoids any
# recurrence of that, now trivially since it no longer reads v2_scale_mode
# either.)
_DOMAIN_MAX_EXPR_LARGE = (
    "length(v2_search) == 0 ? null : "
    "(v2_scale_mode != 'fixed' ? null : v2_name_max_lookup[upper(v2_search)])"
)
COLOR_SMALL = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0),
    # Previously legend=None (true since the very first version of this
    # viz) — but now that the three mini-maps deliberately share one colour
    # scale (to make the real rank-1/2/3 popularity gap visible, see below),
    # leaving them legend-less hid exactly the information that sharing the
    # scale was meant to expose: there was no way to read what a given blue
    # actually corresponds to until clicking through to the large map, whose
    # legend can show a DIFFERENT scale entirely (its own per-name maximum
    # in search mode, not the top-3's shared one). Declaring a legend on
    # each mini-map looks like it should triple it, but because all three
    # share one resolved colour scale (resolve_scale(color="shared") on
    # small_row_v2 below), Vega-Lite merges them into a single legend next
    # to the row, the same way it would for any other shared-scale view.
    legend=alt.Legend(title="For 1 000 births"),
)
# COLOR_LARGE and COLOR_SEARCH share ONE expression (_DOMAIN_MAX_EXPR_LARGE),
# which is what lets resolve_scale(color="shared") on the large map merge
# them into a single legend (declaring two separate scale objects here would
# always render two stacked legends, even when one layer's data is empty —
# see the bug-fix note above). The expression's FIRST check
# (length(v2_search) == 0) is what keeps it from ever leaking a fixed
# numeric ceiling onto the top-3 view: when no search is active, it always
# resolves to null (pure auto-scale) regardless of v2_scale_mode, and the
# fixed/search-specific branch is only ever reached once a search name is
# actually typed.
COLOR_LARGE = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0, domainMax={"expr": _DOMAIN_MAX_EXPR_LARGE}),
    legend=alt.Legend(title="For 1 000 births"),
)
COLOR_SEARCH = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0, domainMax={"expr": _DOMAIN_MAX_EXPR_LARGE}),
    legend=None,
)

# ── 8. Name labels (top-3 panels) ─────────────────────────────────────────────
_MEDALS = {1: "\U0001f947", 2: "\U0001f948", 3: "\U0001f949"}

def name_label_v2(rid, w):
    selected = f"v2_click.rank_id == {rid}"
    medal = _MEDALS[rid]
    return (
        alt.Chart(name_src_v2)
        .mark_text(align="center", baseline="middle", fontSize=13, fontWeight="bold")
        .encode(
            text="label:N",
            x=alt.value(w / 2),
            y=alt.value(12),
            color=alt.condition(selected, alt.value("#1a6bb5"), alt.value("#aaaaaa")),
        )
        .transform_filter(f"datum.annais == v2_year && datum.rank_id == {rid}")
        .transform_calculate("label", f"'{medal} ' + datum.preusuel")
        .properties(width=w, height=22)
    )

# ── 9. Small maps (top-3 panels, unchanged) ───────────────────────────────────
def small_map_v2(rid):
    return (
        alt.Chart(geo_v2)
        .mark_geoshape(stroke="white", strokeWidth=0.4, cursor="pointer")
        .encode(
            color=COLOR_SMALL,
            tooltip=[
                alt.Tooltip("nom:N",  title="Department"),
                alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
            ],
        )
        .transform_calculate("prop", "datum['p_' + v2_year]")
        .transform_filter(f"datum.rank_id == {rid}")
        .add_params(click_sel_v2)
        .project(type="identity", reflectY=True)
        .properties(width=SW, height=SH)
    )

# ── 10. Header above large map ────────────────────────────────────────────────
# Shows either the searched name (if the search box is non-empty) or the
# clicked top-3 name — search takes priority while it has text, which is the
# simplest unambiguous rule (clicking a mini-map again has no effect until
# the search box is cleared). A true "whichever was interacted with last"
# behaviour would need a hand-written Vega event-stream signal outside
# Altair's normal declarative API — judged not worth the added fragility
# for this benefit; see the Visualization-1-style trade-off note below.
large_label_v2 = (
    alt.Chart(name_src_v2)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(LW / 2), y=alt.value(12))
    .transform_filter("length(v2_search) == 0 && datum.annais == v2_year && datum.rank_id == v2_click.rank_id")
    .properties(width=LW, height=22)
)
search_label_v2 = (
    alt.Chart(pd.DataFrame({"label_text": ["placeholder"]}))
    # If the typed name isn't a key in v2_name_max_lookup, the search found no
    # exact match — show an explicit message instead of silently rendering an
    # empty map. Note the wording is deliberately neutral ("no exact match"),
    # not "not in top 1 000": ~13% of the top-1000 names contain an accented
    # character (e.g. STÉPHANE, ANDRÉ), and the search is accent-sensitive
    # (typing "stephane" will not match "STÉPHANE" — see the markdown note
    # on why accents are not stripped). A name typed without its accent would
    # otherwise see a misleading "not in top 1 000" message even though it
    # genuinely is in the list, just spelled differently.
    .transform_calculate(
        label_text="isValid(v2_name_max_lookup[upper(v2_search)]) ? v2_search : 'No exact match — check spelling/accents'"
    )
    .transform_filter("length(v2_search) > 0")
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(
        text="label_text:N",
        x=alt.value(LW / 2),
        y=alt.value(12),
        color=alt.condition(
            "isValid(v2_name_max_lookup[upper(v2_search)])",
            alt.value("black"),
            alt.value("#b3261e"),
        ),
    )
)

# A small subtitle under the searched name, only shown in "fixed" mode, to
# make explicit that the colour ceiling is that SPECIFIC name's own all-time
# peak — not a shared or arbitrary value. Without this, a legend topping out
# at, say, 54‰ for "Charles" looks no different in kind from the unrelated
# 0-324‰ scale used by the top-3 mini-maps just above it; reading the two
# side by side without this label could suggest Charles is uniformly faint
# everywhere, when "auto" mode shows real regional variation (just on a
# scale 6x smaller than the top-3's, since Charles never came close to the
# top-3's historical peak). The number shown is the exact same value driving
# the colour scale itself (v2_name_max_lookup[upper(v2_search)]), so it can
# never drift out of sync with what's actually rendered.
search_subtitle_v2 = (
    alt.Chart(pd.DataFrame({"subtitle_text": ["placeholder"]}))
    .transform_calculate(
        subtitle_text=(
            "length(v2_search) == 0 || !isValid(v2_name_max_lookup[upper(v2_search)]) ? '' : "
            "(v2_scale_mode != 'fixed' ? 'Scale: auto-adjusted to this year’s range' : "
            "'Scale: 0–' + format(v2_name_max_lookup[upper(v2_search)], '.0f') + '‰ — this name’s own all-time peak')"
        )
    )
    .transform_filter("length(v2_search) > 0")
    .mark_text(align="center", baseline="middle", fontSize=10, color="gray", fontStyle="italic")
    .encode(text="subtitle_text:N", x=alt.value(LW / 2), y=alt.value(28))
)

# ── 11. Large map — switches between top-3 selection and free search ─────────
# Both layers share a SINGLE projection, declared once on the combined layer
# below (not on each sub-chart) — declaring .project() separately on each
# geoshape layer let Vega-Lite's projection auto-fit diverge slightly between
# the two (each computing its own fit from whichever data it currently sees),
# which showed up as inconsistent empty space around the map depending on
# which layer was active. A single shared projection avoids that.
large_map_top3_v2 = (
    alt.Chart(geo_v2)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=COLOR_LARGE,
        tooltip=[
            alt.Tooltip("nom:N",  title="Department"),
            alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
        ],
    )
    .transform_calculate("prop", "datum['p_' + v2_year]")
    .transform_filter("length(v2_search) == 0 && datum.rank_id == v2_click.rank_id")
)

large_map_search_v2 = (
    alt.Chart(geo_search_v2)
    # Names are stored in uppercase (matching the INSEE source data), but
    # users naturally type a name in any case (e.g. "Jean", "jean"). Object-key
    # access in Vega's expression language is exact-match, so without
    # normalising the case here, a typed name would silently fail to match
    # its uppercase key and no department would ever light up — exactly the
    # bug we hit when testing with "Jean" before adding upper() below.
    .transform_calculate(name_entry="datum.names_data[upper(v2_search)]")
    .transform_filter("length(v2_search) > 0 && isValid(datum.name_entry)")
    .transform_calculate(idx="indexof(datum.name_entry.years, v2_year)")
    # IMPORTANT: a department with no recorded births for this name in this
    # EXACT year (idx == -1, because the sparse years/props lists only store
    # years with non-zero counts) is not the same thing as "this name is
    # absent from the dataset". The first version filtered out idx == -1
    # rows entirely, which made those departments disappear from the map
    # (pure white, outside the colour scale) instead of showing a real zero
    # (very pale blue, inside the scale) — visible as an unexplained
    # patchwork of blank departments when compared side-by-side with the
    # top-3 panels, which never have this issue because their wide pivot
    # table is filled with explicit zeros for every year. Mapping a missing
    # year to 0 here (instead of filtering the row out) makes the search
    # overlay consistent with the top-3 panels.
    .transform_calculate("prop", "datum.idx >= 0 ? datum.name_entry.props[datum.idx] : 0")
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=COLOR_SEARCH,
        tooltip=[
            alt.Tooltip("nom:N",  title="Department"),
            alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
        ],
    )
)

large_map_v2 = (
    alt.layer(large_map_top3_v2, large_map_search_v2)
    .resolve_scale(color="shared")
    .add_params(year_p_v2, search_p_v2, scale_mode_p_v2, name_max_lookup_p_v2)
    .project(type="identity", reflectY=True)
    .properties(
        width=LW, height=LH,
        title=alt.TitleParams(
            text=alt.ExprRef("toString(v2_year)"),
            orient="bottom",
            anchor="end",
            fontSize=36,
            fontWeight="bold",
            color="#999999",
            offset=4,
            dx=130,
        ),
    )
)

# ── 12. Assemble ──────────────────────────────────────────────────────────────
labels_row_v2 = alt.hconcat(
    name_label_v2(1, SW), name_label_v2(2, SW), name_label_v2(3, SW), spacing=2
)
small_row_v2 = alt.hconcat(
    small_map_v2(1), small_map_v2(2), small_map_v2(3), spacing=10
).resolve_scale(color="shared")
# Follows the same auto/fixed toggle as the large map (one shared
# COLOR_SMALL definition, see step 7): in "auto" mode the three panels now
# rescale together to their combined min/max, rather than each optimising
# its own contrast independently — which previously could make the rank-3
# name look just as "dark" as rank-1, hiding the very popularity gap the
# medal ordering is meant to show. In "fixed" mode this changes nothing
# visually (all three already shared the same all-time maximum either way).

separator_v2 = (
    alt.Chart(pd.DataFrame({"y": [0]}))
    .mark_rule(color="#222222", strokeWidth=1.5)
    .encode(y=alt.value(1))
    .properties(width=SW * 3 + 15, height=4)
)

note_v2 = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_text(align="center", baseline="middle", fontSize=11, color="gray", fontStyle="italic")
    .encode(text=alt.value("Note: click a mini map to view it large, or type a name to search (top 1 000 names; clear the box to go back to the top-3)"),
            x=alt.value(LW / 2), y=alt.value(9))
    .properties(width=LW, height=18)
)

large_header_row_v2 = (
    alt.layer(large_label_v2, search_label_v2, search_subtitle_v2)
    .properties(width=LW, height=34)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    v2_new = (
        alt.vconcat(
            labels_row_v2,
            small_row_v2,
            separator_v2,
            note_v2,
            large_header_row_v2,
            large_map_v2,
            spacing=6,
        )
        .resolve_scale(color="independent")
        .configure_view(strokeWidth=0)
    )

v2_new


## Visualization 3 — Is there a gender effect on unisex names? (V2)

**Design (updated after peer feedback):** Same butterfly chart layout — boys' bars left, girls' bars right, one row per year — extended with two independent additions:

- **Raw / percent toggle:** a radio switches each bar's length between the raw birth count and that sex's share of the name's total births *that year* (so the two bars for a given year always sum to 100%). We tested an alternative — normalising each sex's count by the national total for that sex that year — and rejected it: for a heavily lopsided name like Jean (boys outnumber girls by up to ~6800:1 in some years), that alternative produces tiny percentages on *both* sides (Jean represents well under 1% of all boys born most years), reproducing the exact "invisible bar" problem at a different numeric scale instead of solving it. Normalising by the name's own yearly total instead guarantees the smaller share is visible whenever it is genuinely non-zero — verified on "Claude" (a real ~88%/12% split in 1940 reads as two clearly visible bars in percent mode, instead of two near-invisible slivers under the rejected alternative).
- **Mini area chart of total yearly popularity**, placed above the butterfly and sharing its year-range filter, answers the complementary question the butterfly alone cannot: is this name dying out, or shifting its gender association while remaining just as popular? Tested on "Charlie": the butterfly shows a name that was ~85% boys in 2000 and ~70% girls by 2020 — a genuine gender-association shift — while the area chart above shows its total popularity only growing throughout, ruling out "the name is just fading" as an alternative explanation a butterfly-only view couldn't distinguish.

**What we kept on purpose:** the year-range and step sliders, and raw counts as the default display mode (percent is an added lens, not a replacement) — none of these were contested in feedback.

**Strengths:** the toggle makes every name in the top-50 readable regardless of how lopsided it is; the area chart resolves the "dying vs. shifting" ambiguity that the original design's authors flagged as impossible to answer from the butterfly alone.
**Trade-offs:** in percent mode, a sex with literally zero births that year still renders as an invisible bar (correctly — zero is zero, not a scaling artefact) — verified directly against the underlying data rather than assumed; the area chart's y-axis is not interactive (no toggle of its own), kept deliberately simple since its only job here is to show a trend at a glance alongside the butterfly, not to be a second fully-featured chart.

**Bug fix — unreadable x-axis on the mini area chart.** The first version hid the area chart's x-axis entirely (no labels, no ticks), on the assumption that the butterfly's year axis just below would serve as enough of a reference. That assumption was wrong: the butterfly's years run vertically along its y-axis, while the area chart's years run horizontally along its own x-axis — they are not the same visual axis, so hiding it left the area chart with no readable time reference of its own (a reader correctly couldn't tell what the horizontal position meant, only what appeared to be the unrelated axis values from the chart below bleeding through visually). Fixed by giving the area chart its own labelled, titled year axis, with the tick count capped so a full 1900-2020 span doesn't try to cram over a hundred year labels into a 600px strip.

In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
nat_sex_v3 = df.groupby(["preusuel", "annais", "sexe"], as_index=False)["nombre"].sum()

# Unisex names: given to both sexes with ≥ 200 total births per sex
sex_tot_v3 = nat_sex_v3.groupby(["preusuel", "sexe"])["nombre"].sum().reset_index()
sex_piv_v3 = sex_tot_v3.pivot(index="preusuel", columns="sexe", values="nombre").fillna(0)
dual_v3    = sex_piv_v3[(sex_piv_v3[1] >= 200) & (sex_piv_v3[2] >= 200)].index.tolist()
dual_sorted_v3 = (
    df[df.preusuel.isin(dual_v3)]
    .groupby("preusuel")["nombre"].sum()
    .sort_values(ascending=False)
    .head(50).index.tolist()
)
_pref_v3   = ["DOMINIQUE", "CAMILLE", "CLAUDE"]
default_v3 = next((n for n in _pref_v3 if n in dual_sorted_v3), dual_sorted_v3[0])
data_v3    = nat_sex_v3[nat_sex_v3.preusuel.isin(dual_sorted_v3)].copy()
name_df_v3 = pd.DataFrame({"preusuel": dual_sorted_v3})
print(f"Unisex names in dropdown: {len(dual_sorted_v3)}.  Default: {default_v3}")

# Pre-aggregate, per name, the TOTAL births (both sexes combined) per year —
# this is what the new mini line chart shows. Computed once in Python rather
# than via a Vega-side aggregate transform purely for clarity/consistency
# with the rest of the notebook's style; the dataset is tiny (50 names x up
# to 121 years) so there's no performance reason either way.
totals_v3 = (
    data_v3.groupby(["preusuel", "annais"], as_index=False)["nombre"]
    .sum()
    .rename(columns={"nombre": "year_total"})
)

# ── Params ────────────────────────────────────────────────────────────────────
name_p3 = alt.param(
    name="v3_name",
    bind=alt.binding_select(options=dual_sorted_v3, name="First name: "),
    value=default_v3,
)
ymin_p3 = alt.param(
    name="v3_ymin",
    bind=alt.binding_range(min=1900, max=2019, step=1, name="From year: "),
    value=1970,
)
ymax_p3 = alt.param(
    name="v3_ymax",
    bind=alt.binding_range(min=1901, max=2020, step=1, name="To year: "),
    value=2020,
)
step_p3 = alt.param(
    name="v3_step",
    bind=alt.binding_range(min=1, max=20, step=1, name="Step (years): "),
    value=5,
)
# New: raw counts vs percent-of-name-total toggle. In "percent" mode each
# pair of bars for a given year always sums to 100% of that year's births
# of this name — chosen over normalising by each sex's national total
# (which we tested and rejected: for a heavily lopsided name like Jean,
# that alternative produces tiny percentages on BOTH sides, since even the
# dominant sex's count is a small share of all boys born that year — it
# reproduces the same "invisible bar" problem at a different scale instead
# of solving it). Normalising by the name's own yearly total guarantees the
# smaller share is visible whenever it is non-zero, which is what actually
# answers "is this name balanced or lopsided this year".
mode_p3 = alt.param(
    name="v3_mode",
    bind=alt.binding_radio(options=["raw", "percent"], name="Display: "),
    value="raw",
)

# ── Chart ─────────────────────────────────────────────────────────────────────
# Dynamic name header
name_title_v3 = (
    alt.Chart(name_df_v3)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(300), y=alt.value(12))
    .transform_filter("datum.preusuel === v3_name")
    .properties(width=600, height=22)
)

# New: mini line chart of total yearly popularity (both sexes combined),
# placed above the butterfly and sharing its year filters (range + step),
# so the two views always describe the same window. This directly answers
# the complementary question the butterfly alone cannot: is this name
# dying out, or merely shifting its gender balance while staying popular
# overall? A name can show a dramatic colour-balance shift in the butterfly
# while its total bar here barely moves (a genuine gender-association
# change) — or the reverse, a shrinking total with a stable balance (the
# name is simply falling out of fashion for everyone). Neither reading was
# possible from the butterfly's raw left/right split alone.
total_line_v3 = (
    alt.Chart(totals_v3)
    .mark_area(line={"color": "#555555", "strokeWidth": 1.5}, color="#dddddd", opacity=0.6)
    .encode(
        x=alt.X(
            "annais:Q",
            title="Year",
            scale=alt.Scale(domain={"expr": "[v3_ymin, v3_ymax]"}),
            # The x-axis was previously hidden entirely (labels=False), on
            # the assumption that the butterfly's year axis just below would
            # be enough of a reference. That assumption didn't hold: the
            # butterfly's years run vertically (its y-axis), while this
            # chart's years run horizontally (its x-axis) — they are not the
            # same visual axis, so hiding this one left the area chart with
            # no readable time reference at all. tickCount is capped so a
            # full 1900-2020 span doesn't try to cram 120+ year labels into
            # a 600px-wide strip.
            axis=alt.Axis(format="d", tickCount=8, labelAngle=0, grid=True),
        ),
        y=alt.Y("year_total:Q", title="Total births"),
        tooltip=[
            alt.Tooltip("annais:Q", title="Year", format="d"),
            alt.Tooltip("year_total:Q", title="Total births", format=","),
        ],
    )
    .transform_filter("datum.preusuel === v3_name")
    .transform_filter("datum.annais >= v3_ymin && datum.annais <= v3_ymax")
    .properties(width=600, height=70)
)

# Horizontal butterfly bars
# Boys  → val = -magnitude  (extends left)
# Girls → val = +magnitude  (extends right)
# magnitude is either the raw birth count, or that sex's share of this
# name's total births THAT YEAR (always summing to 100% across both bars).
bars_v3 = (
    alt.Chart(data_v3)
    .mark_bar()
    .encode(
        x=alt.X(
            "val:Q",
            title="← Boys | Girls →",
            axis=alt.Axis(
                labelExpr="v3_mode == 'percent' ? format(abs(datum.value), '.0f') + '%' : format(abs(datum.value), ',.0f')",
                labelAngle=0,
            ),
        ),
        y=alt.Y(
            "annais:O",
            title="Year",
            sort="ascending",
            axis=alt.Axis(labelAngle=0),
        ),
        color=alt.Color(
            "gender:N",
            scale=alt.Scale(domain=["Boys", "Girls"], range=["#4e78a8", "#e7729a"]),
            legend=alt.Legend(title="Sex"),
        ),
        tooltip=[
            alt.Tooltip("annais:O",  title="Year"),
            alt.Tooltip("gender:N",  title="Sex"),
            alt.Tooltip("nombre:Q",  title="Births", format=","),
            alt.Tooltip("pct:Q",     title="% of that year's births", format=".1f"),
        ],
    )
    .transform_filter("datum.preusuel === v3_name")
    .transform_filter("datum.annais >= v3_ymin && datum.annais <= v3_ymax")
    .transform_filter("(datum.annais - v3_ymin) % v3_step === 0")
    .transform_joinaggregate(year_total="sum(nombre)", groupby=["annais"])
    .transform_calculate("pct", "datum.year_total > 0 ? datum.nombre / datum.year_total * 100 : 0")
    .transform_calculate("magnitude", "v3_mode == 'percent' ? datum.pct : datum.nombre")
    .transform_calculate("val",    "datum.sexe === 1 ? -datum.magnitude : datum.magnitude")
    .transform_calculate("gender", "datum.sexe === 1 ? 'Boys' : 'Girls'")
    .properties(width=600, height=400)
)

# Vertical centre line at x = 0
centre_v3 = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="#333333", strokeWidth=1.5)
    .encode(x=alt.X("x:Q"))
)

v3 = (
    alt.vconcat(
        name_title_v3,
        total_line_v3,
        alt.layer(bars_v3, centre_v3),
        spacing=4,
    )
    .add_params(name_p3, ymin_p3, ymax_p3, step_p3, mode_p3)
    .properties(
        title=alt.TitleParams(
            "Gender butterfly chart - Boys vs Girls",
            subtitle="Top-50 unisex names by total popularity (1900-2020). Toggle raw counts / percent-of-year-total to read lopsided names; the area chart above tracks the name's overall popularity.",
            subtitleFontSize=11,
            subtitleColor="gray",
        )
    )
    .configure_view(strokeWidth=0)
)

v3
